# EEG for LLMs: A Telemetry Layer for Online Uncertainty Monitoring and Decision Policies

**Author:** Nikolay Yudin  
**Email:** n.yudin@gmail.com  
**Date:** 2026-01-30  
**Code:** https://github.com/nick-yudin/grokking-research  

This notebook runs the Paper 1 monitoring-only experiment on Wikipedia continuation (no gold labels).

## Abstract
Token streams are a human-oriented interface that can obscure generation dynamics and encourage brittle analyses (e.g., relying on chain-of-thought text). We introduce an "EEG-like" telemetry layer for autoregressive decoding that records lightweight internal signals during generation—uncertainty, surprisal, distribution shift, and sparse layer summaries—yielding real-time traces of model state evolution without parsing chain-of-thought text. Across three model families and three task types (27 runs = 3 models × 3 tasks × 3 seeds), we find that telemetry signatures vary strongly across models and tasks, and that early-window uncertainty can predict failures above random on labeled tasks. As an application demo, we show how telemetry can gate a simple cascade policy (accept / retry / route) on a Llama-8B → Qwen-14B pair.


In [ ]:
# Install runtime dependencies
# Note: this notebook assumes internet access for pip and model downloads.

!pip -q install -U "transformers>=4.46" "datasets>=2.20" "accelerate" "matplotlib" "pandas" "tqdm" "seaborn"

import os
from pathlib import Path

REPO_DIR = Path('/content/grokking-research')
assert REPO_DIR.exists(), f"Repo not found at {REPO_DIR}. Upload and extract the repo to /content/grokking-research."

os.chdir(str(REPO_DIR))
print('Repo:', REPO_DIR)


In [ ]:
import subprocess
from pathlib import Path

# Where we store outputs (no Drive; local to this Colab runtime).
ARTIFACTS_ROOT = str(Path('/content') / 'artifacts')
Path(ARTIFACTS_ROOT).mkdir(parents=True, exist_ok=True)

def run(cmd: str):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True)

# Experiment config (edit as needed)
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"
SEED = 0
N = 50  # Increase to 200 to match the paper's canonical runs.
RUN_ID = f"paper1_wiki_example.qwen14b.seed{SEED}.n{N}"

print('Artifacts root:', ARTIFACTS_ROOT)
print('RUN_ID:', RUN_ID)


In [ ]:
# Run Wikipedia continuation with monitoring-only early-stop (no gold labels).
run(
    " ".join(
        [
            "python3 -u latent_configuration/01_interpretable_latent_probes/telemetry_collect.py",
            f"--run_id {RUN_ID}",
            f"--artifacts_root {ARTIFACTS_ROOT}",
            f"--model_name \"{MODEL_NAME}\"",
            "--dataset wikipedia --trust_remote_code",
            "--wikipedia_config 20231101.en --wikipedia_split train",
            "--wikipedia_prompt_chars 600",
            "--wikipedia_streaming --wikipedia_shuffle_buffer 10000",
            f"--num_examples {N} --seed {SEED}",
            "--max_new_tokens 256",
            "--dtype bf16 --topk 50 --layers auto:last4",
            "--temperature 0.0",
            # Early stop: high uncertainty AND instability (entropy + delta_l1)
            "--stop_uncertainty_when high --stop_uncertainty_logic and",
            "--stop_entropy_norm_ge 0.33 --stop_delta_l1_ge 0.55",
            "--stop_window 16 --stop_patience 3 --stop_min_new_tokens 48",
            "--sample_log_every 0 --heartbeat_s 900",
        ]
    )
)


In [ ]:
run(
    " ".join(
        [
            "python3 -u latent_configuration/01_interpretable_latent_probes/telemetry_analyze.py",
            f"--run_dir {ARTIFACTS_ROOT}/runs/{RUN_ID}",
        ]
    )
)


In [ ]:
from pathlib import Path

report_path = Path(ARTIFACTS_ROOT) / 'runs' / RUN_ID / 'analysis' / 'report.md'
print('report:', report_path)
print(report_path.read_text(encoding='utf-8')[:4000])


## Download artifacts

Use the file browser to download:
- `/content/artifacts/runs/<RUN_ID>/analysis/` (reports)
- `/content/artifacts/runs/<RUN_ID>/telemetry.jsonl` (raw telemetry)
- `/content/artifacts/runs/<RUN_ID>/samples.jsonl` (per-sample records)
